In [ ]:
import time
import sys
import tomoDataClass
from datetime import datetime
from helperFunctions import DualLogger, convert_to_tiff, subpixel_shift, degree_to_positiveRadians, runwidget
import tomopy
import matplotlib.pyplot as plt
import matplotlib

In [ ]:


# -------------------------
# CONFIGURATION FLAGS
# -------------------------
log = False           # Set to True to enable logging output to a file
saveToFile = False     # Set to True to save aligned projection data to a TIFF file
reconstruct = False     # Set to True to save the reconstruction to a TIFF file

# -------------------------
# SETUP: Timing & Logging
# -------------------------
start_time = time.time()  # Start execution timer
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")  # Generate timestamp for filenames
print("timestamp:", timestamp)

if log:
    # Redirect stdout/stderr to both console and log file
    sys.stdout = DualLogger(f'logs/output_tomoMono_align{timestamp}.txt', 'w')

print("Running Image Registration Script")


In [ ]:


#Importing data from Taylor Buckway h5 file (APS data)
import h5py
import numpy as np
# filename = r"/home/ljh79/TomoMono/data/noglow_tomo_128.hdf5"
# filename = '/home/ljh79/groups/grp_ptychi/nobackup/autodelete/Oct2025APSdata/tomo_data_run_final.hdf5'
filename = "/Users/levihancock/Library/CloudStorage/Box-Box/BYU_CXI_Research_Team/ProjectFolders/IFE-STAR/IFE-Ptycho-Tomo/APS_2ID_GUP1013052_August_2025/levi_tomoReconstructions"

with h5py.File(filename, "r") as f:
    data = np.array(f["/data"])[::2,::2,::2]
    angles = degree_to_positiveRadians(list(f["/angles"]))

print("data shape is: ", data.shape)
print("angles shape is: ", len(angles))

runwidget(data)

tomo = tomoDataClass.tomoData(data, angles)
scale_info = None

In [ ]:

print("Starting alignment")

savePath = f"/home/ljh79/TomoMono/alignedProjections/APSbeamtime_Oct25/aligned_XC_summed_{timestamp}.tif"
print("\n\nCreating aligned projections:", savePath)

# # Ensure alignment begins from original, unmodified projections
# tomo.reset_workingProjections(x_size = data.shape[2], y_size=data.shape[1])
tomo.reset_workingProjections(x_size = 512+256, y_size=256, cropTopCenter=True) #if halfing input
# tomo.reset_workingProjections(x_size = 1024+512, y_size=512, cropTopCenter=True)


# -------------------------
# ALIGNMENT STRATEGY
# -------------------------
# Choose and configure alignment algorithm below:
tomo.shift_min_to_middle()
tomo.cross_correlate_align_to_sum(tolerance=0.01, max_iterations = 15, yROI_Range=[0, tomo.workingProjections.shape[1]], xROI_Range=[0, tomo.workingProjections.shape[2]], maxShiftTolerance=5)
# tomo.PMA(max_iterations = 5, tolerance=0.0001, algorithm="SIRT_CUDA", crop_bottom_center_y = tomo.workingProjections.shape[1]-80, crop_bottom_center_x = 1200, isPhaseData = True)
tomo.center_projections()


# # Choose and configure alignment algorithm below:
# tomo.shift_min_to_middle()
# tomo.cross_correlate_align_to_sum(tolerance=0.01, max_iterations = 10, yROI_Range=[0, tomo.workingProjections.shape[1]], xROI_Range=[0, tomo.workingProjections.shape[2]], maxShiftTolerance=5)
# tomo.PMA(max_iterations = 5, tolerance=0.0001, algorithm="SIRT_CUDA", crop_bottom_center_y = tomo.workingProjections.shape[1]-80, crop_bottom_center_x = 1200, isPhaseData = True)
# tomo.center_projections()

# Apply the computed shifts to original data to finalize alignment
tomo.make_updates_shift()

runwidget(tomo.get_finalProjections())
# -------------------------
# SAVE RESULTS (Optional)
# -------------------------
if saveToFile:
    convert_to_tiff(tomo.get_finalProjections(), savePath, scale_info)
if reconstruct:
    tomo.reconstruct(algorithm="SIRT_CUDA", snr_db=None)
    recon_path = f"reconstructions/APSbeamtime_Oct25/recon_XC_{timestamp}.tif"
    convert_to_tiff(tomo.get_recon(), recon_path, scale_info)